# PEFT 库 LoRA 实战 - OpenAI Whisper-large-v2

本教程使用 LoRA 在`OpenAI Whisper-large-v2`模型上实现`语音识别(ASR)`任务的微调训练。

同时，我们还结合了`int8` 量化进一步降低训练过程资源开销，同时保证了精度几乎不受影响。

![LoRA](https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/peft/lora_diagram.png)

本教程主要训练流程如下：
- 全局参数设置
- 数据准备
    - 下载数据集：训练、验证和评估集
    - 预处理数据：降采样、移除不必要字段等
    - 数据抽样（演示需要）
    - 应用数据集处理（`Dataset.map`）
    - 自定义语音数据处理器
- 模型准备
    - 加载和处理 `int8` 精度 Whisper-Large-v2 模型
    - LoRA Adapter 参数配置
    - 实例化 PEFT Model：`peft_model = get_peft_model(model, config)`
- 模型训练
    - 训练参数配置 Seq2SeqTrainingArguments
    - 实例化训练器 Seq2SeqTrainer
    - 训练模型
    - 保存模型
- 模型推理
    - 使用 `PeftModel` 加载 LoRA 微调后 Whisper 模型
    - 使用 `Pipeline API` 部署微调后 Whisper 实现中文语音识别任务

## 全局参数设置

In [1]:
model_name_or_path = "openai/whisper-large-v2"
model_dir = "../models/whisper-large-v2-asr-int8"
dataset_dir="~/LLM-quickstart/models/datasets/voice/cv-corpus-25.0-2026-03-09/zh-CN"

language = "Chinese (China)"
language_abbr = "zh-CN"
task = "transcribe"
dataset_name = "mozilla-foundation/common_voice_11_0"

batch_size=32*3

## 数据准备

### 下载数据集 Common Voice

Common Voice 11.0 数据集包含许多不同语言的录音，总时长达数小时。

本教程以中文数据为例，展示如何使用 LoRA 在 Whisper-large-v2 上进行微调训练。

首先，初始化一个DatasetDict结构，并将训练集（将训练+验证拆分为训练集）和测试集拆分好，按照中文数据集构建配置加载到内存中：

In [2]:
from datasets import load_dataset, DatasetDict, Audio, Features, Value
import os, shutil

dataset_dir = "~/LLM-quickstart/models/datasets/voice/cv-corpus-25.0-2026-03-09/zh-CN"
dataset_dir = os.path.expanduser(dataset_dir)
clips_dir = os.path.join(dataset_dir, "clips")
cache_dir = os.path.expanduser("~/.cache/huggingface/datasets")
if os.path.exists(cache_dir):
    shutil.rmtree(cache_dir)

common_voice = DatasetDict()

features = Features({
    "client_id": Value("string"),
    "path": Value("string"),
    "sentence_id": Value("string"),
    "sentence": Value("string"),
    "sentence_domain": Value("string"),
    "up_votes": Value("string"),
    "down_votes": Value("string"),
    "age": Value("string"),
    "gender": Value("string"),
    "accents": Value("string"),
    "variant": Value("string"),
    "locale": Value("string"),
    "segment": Value("string"),
})

ds = load_dataset(
    "csv",
    data_files={
        "train": os.path.join(dataset_dir, "train.tsv"),
        "validation": os.path.join(dataset_dir, "dev.tsv"),
        "test": os.path.join(dataset_dir, "test.tsv"),
    },
    delimiter="\t",
    features=features,
)

def _add_audio_path(ex):
    return {"audio": os.path.join(clips_dir, ex["path"])}

ds["train"] = ds["train"].map(_add_audio_path)
ds["validation"] = ds["validation"].map(_add_audio_path)
ds["test"] = ds["test"].map(_add_audio_path)

ds["train"] = ds["train"].cast_column("audio", Audio())
ds["validation"] = ds["validation"].cast_column("audio", Audio())
ds["test"] = ds["test"].cast_column("audio", Audio())

common_voice["train"] = ds["train"]
common_voice["validation"] = ds["validation"]
common_voice["test"] = ds["test"]


Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/29608 [00:00<?, ? examples/s]

Map:   0%|          | 0/10653 [00:00<?, ? examples/s]

Map:   0%|          | 0/10653 [00:00<?, ? examples/s]

In [3]:
common_voice

DatasetDict({
    train: Dataset({
        features: ['client_id', 'path', 'sentence_id', 'sentence', 'sentence_domain', 'up_votes', 'down_votes', 'age', 'gender', 'accents', 'variant', 'locale', 'segment', 'audio'],
        num_rows: 29608
    })
    validation: Dataset({
        features: ['client_id', 'path', 'sentence_id', 'sentence', 'sentence_domain', 'up_votes', 'down_votes', 'age', 'gender', 'accents', 'variant', 'locale', 'segment', 'audio'],
        num_rows: 10653
    })
    test: Dataset({
        features: ['client_id', 'path', 'sentence_id', 'sentence', 'sentence_domain', 'up_votes', 'down_votes', 'age', 'gender', 'accents', 'variant', 'locale', 'segment', 'audio'],
        num_rows: 10653
    })
})

In [4]:
common_voice["train"].shuffle(seed=31)[:1]

{'client_id': ['25bc975d06200b7b1c9135db090561cb0d9b28d172e51c6826f9e665a3ba7bddda6ec0c6c09102f5d1d4ec99cd094b9c91ad0141e49341246bc6eb2e7167f2e7'],
 'path': ['common_voice_zh-CN_19712014.mp3'],
 'sentence_id': ['358ad683a5e5f8bf2409b60a14100624748e22e874234e6e03c3a6f708ede9e4'],
 'sentence': ['此地俄语地名来源俄国海军上将。'],
 'sentence_domain': [None],
 'up_votes': ['2'],
 'down_votes': ['0'],
 'age': ['thirties'],
 'gender': ['female_feminine'],
 'accents': ['出生地：31 上海市'],
 'variant': [None],
 'locale': ['zh-CN'],
 'segment': [None],
 'audio': [{'path': '/home/zhanguo/LLM-quickstart/models/datasets/voice/cv-corpus-25.0-2026-03-09/zh-CN/clips/common_voice_zh-CN_19712014.mp3',
   'array': array([ 0.00000000e+00,  1.89695308e-12,  8.78513191e-13, ...,
          -9.13534532e-05, -1.35976792e-04,  2.77700346e-05]),
   'sampling_rate': 48000}]}

## 预处理训练数据集


In [5]:
from transformers import AutoFeatureExtractor, AutoTokenizer, AutoProcessor

# 从预训练模型加载特征提取器
feature_extractor = AutoFeatureExtractor.from_pretrained(model_name_or_path)

# 从预训练模型加载分词器，可以指定语言和任务以获得最适合特定需求的分词器配置
tokenizer = AutoTokenizer.from_pretrained(model_name_or_path, language=language, task=task)

# 从预训练模型加载处理器，处理器通常结合了特征提取器和分词器，为特定任务提供一站式的数据预处理
processor = AutoProcessor.from_pretrained(model_name_or_path, language=language, task=task)

/data00/home/zhanguo/miniforge3/envs/py310/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


#### 移除数据集中不必要的字段

In [6]:
common_voice=common_voice.remove_columns("sentence_id")

In [7]:
common_voice = common_voice.remove_columns(
    ["age", "client_id", "down_votes", "gender", "locale", "path", "segment", "up_votes","accents","variant","sentence_domain"]
)

In [8]:
common_voice["train"][0]

{'sentence': '目前正在爆发的兹卡病毒疫情引发运动员和游客的担忧。',
 'audio': {'path': '/home/zhanguo/LLM-quickstart/models/datasets/voice/cv-corpus-25.0-2026-03-09/zh-CN/clips/common_voice_zh-CN_22154274.mp3',
  'array': array([ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
         -1.62431343e-06, -6.40250619e-07,  1.10271367e-06]),
  'sampling_rate': 48000}}

#### 降采样音频数据

查看`common_voice` 数据集介绍，你会发现其音频是以48kHz的采样率进行采样的.

而`Whisper`模型是在16kHZ的音频输入上预训练的，因此我们需要将音频输入降采样以匹配模型预训练时使用的采样率。

通过在音频列上使用`cast_column`方法，并将`sampling_rate`设置为16kHz来对音频进行降采样。

下次调用时，音频输入将实时重新取样：

In [9]:
from datasets import Audio

common_voice = common_voice.cast_column("audio", Audio(sampling_rate=16000))

In [10]:
# sampling_rate 从 48KHZ 降为 16KHZ
common_voice["train"][0]

{'sentence': '目前正在爆发的兹卡病毒疫情引发运动员和游客的担忧。',
 'audio': {'path': '/home/zhanguo/LLM-quickstart/models/datasets/voice/cv-corpus-25.0-2026-03-09/zh-CN/clips/common_voice_zh-CN_22154274.mp3',
  'array': array([ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
          3.18077582e-06,  3.89223715e-07, -1.19029482e-06]),
  'sampling_rate': 16000}}

### 整合以上数据处理为一个函数

该数据预处理函数应该包括：
- 通过加载音频列将音频输入重新采样为16kHZ。
- 使用特征提取器从音频数组计算输入特征。
- 将句子列标记化为输入标签。

In [11]:
def prepare_dataset(batch):
    audio = batch["audio"]
    batch["input_features"] = feature_extractor(audio["array"], sampling_rate=audio["sampling_rate"]).input_features[0]
    batch["labels"] = tokenizer(batch["sentence"]).input_ids
    return batch

In [12]:
common_voice

DatasetDict({
    train: Dataset({
        features: ['sentence', 'audio'],
        num_rows: 29608
    })
    validation: Dataset({
        features: ['sentence', 'audio'],
        num_rows: 10653
    })
    test: Dataset({
        features: ['sentence', 'audio'],
        num_rows: 10653
    })
})

### 数据抽样（演示需要）

在 Whisper-Large-v2 上使用小规模数据进行演示训练，保持以下训练参数不变（batch_size=64）。

使用 640 个样本训练，320个样本验证和评估，恰好使得1个 epoch 仅需10 steps 即可完成训练。

（在 NVIDIA T4 上需要10-15分钟）

In [13]:
small_common_voice = DatasetDict()

small_common_voice["train"] = common_voice["train"].shuffle(seed=16).select(range(640))
small_common_voice["validation"] = common_voice["validation"].shuffle(seed=16).select(range(320))
small_common_voice["test"] = common_voice["test"].shuffle(seed=16).select(range(320))

In [14]:
small_common_voice

DatasetDict({
    train: Dataset({
        features: ['sentence', 'audio'],
        num_rows: 640
    })
    validation: Dataset({
        features: ['sentence', 'audio'],
        num_rows: 320
    })
    test: Dataset({
        features: ['sentence', 'audio'],
        num_rows: 320
    })
})

### 如果全量训练，则使用完整数据代替抽样

In [15]:
# 抽样数据处理
tokenized_common_voice = small_common_voice.map(prepare_dataset)

# 完整数据训练，尝试开启 `num_proc=8` 参数多进程并行处理（如阻塞无法运行，则不使用此参数）
# tokenized_common_voice = common_voice.map(prepare_dataset, num_proc=8)

Map:   0%|          | 0/640 [00:00<?, ? examples/s]

Map:   0%|          | 0/320 [00:00<?, ? examples/s]

Map:   0%|          | 0/320 [00:00<?, ? examples/s]

In [16]:
tokenized_common_voice

DatasetDict({
    train: Dataset({
        features: ['sentence', 'audio', 'input_features', 'labels'],
        num_rows: 640
    })
    validation: Dataset({
        features: ['sentence', 'audio', 'input_features', 'labels'],
        num_rows: 320
    })
    test: Dataset({
        features: ['sentence', 'audio', 'input_features', 'labels'],
        num_rows: 320
    })
})

### 自定义语音数据整理器

定义了一个针对语音到文本（Seq2Seq）模型的自定义数据整理器类，特别适用于输入为语音特征、输出为文本序列的数据集。


这个整理器（`DataCollatorSpeechSeq2SeqWithPadding`）旨在将数据点批量打包，将每个批次中的`attention_mask`填充到最大长度，以保持批处理中张量形状的一致性，并用`-100`替换填充值，以便在损失函数中被忽略。这对于神经网络的高效训练至关重要。

In [17]:
import torch

from dataclasses import dataclass
from typing import Any, Dict, List, Union

# 定义一个针对语音到文本任务的数据整理器类
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any  # 处理器结合了特征提取器和分词器

    # 整理器函数，将特征列表处理成一个批次
    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # 从特征列表中提取输入特征，并填充以使它们具有相同的形状
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        # 从特征列表中提取标签特征（文本令牌），并进行填充
        label_features = [{"input_ids": feature["labels"]} for feature in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        # 使用-100替换标签中的填充区域，-100通常用于在损失计算中忽略填充令牌
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        # 如果批次中的所有序列都以句子开始令牌开头，则移除它
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        # 将处理过的标签添加到批次中
        batch["labels"] = labels

        return batch  # 返回最终的批次，准备好进行训练或评估

In [18]:
# 用给定的处理器实例化数据整理器
data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

## 模型准备

### 加载预训练模型（int8 精度）

使用 `int8 ` 精度加载预训练模型，进一步降低显存需求。

In [19]:
from transformers import AutoModelForSpeechSeq2Seq

model = AutoModelForSpeechSeq2Seq.from_pretrained(model_name_or_path, load_in_8bit=True, device_map="auto")

/data00/home/zhanguo/miniforge3/envs/py310/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [20]:
# 设置模型配置中的forced_decoder_ids属性为None
model.config.forced_decoder_ids = None  # 这通常用于指定在解码（生成文本）过程中必须使用的特定token的ID，设置为None表示没有这样的强制要求

# 设置模型配置中的suppress_tokens列表为空
model.config.suppress_tokens = []  # 这用于指定在生成过程中应被抑制（不生成）的token的列表，设置为空列表表示没有要抑制的token

### PEFT 微调前的模型处理

在使用 `peft` 训练 int8 模型之前，需要进行一些预处理：
- 将所有非 `int8` 精度模块转换为全精度（`fp32`）以保证稳定性
- 为输入嵌入层添加一个 `forward_hook`，以启用输入隐藏状态的梯度计算
- 启用梯度检查点以实现更高效的内存训练

使用 `peft` 库预定义的工具函数 `prepare_model_for_int8_training`，便可自动完成以上模型处理工作。

In [21]:
from peft import prepare_model_for_int8_training

model = prepare_model_for_int8_training(model)

/data00/home/zhanguo/miniforge3/envs/py310/lib/python3.10/site-packages/peft/utils/other.py:141: FutureWarning: prepare_model_for_int8_training is deprecated and will be removed in a future version. Use prepare_model_for_kbit_training instead.
  warnings.warn(


### LoRA Adapter 配置

在 `peft` 中使用`LoRA`非常简捷，借助 `PeftModel`抽象，我们可以快速使用低秩适配器（LoRA）到任意模型。

通过使用 `peft` 中的 `get_peft_model` 工具函数来实现。

#### 关于 LoRA 超参数的说明：
```
MatMul(B,A) * Scaling
Scaling = LoRA_Alpha / Rank
```

In [22]:
from peft import LoraConfig, PeftModel, LoraModel, LoraConfig, get_peft_model

# 创建一个LoraConfig对象，用于设置LoRA（Low-Rank Adaptation）的配置参数
config = LoraConfig(
    r=4,  # LoRA的秩，影响LoRA矩阵的大小
    lora_alpha=64,  # LoRA适应的比例因子
    # 指定将LoRA应用到的模型模块，通常是attention和全连接层的投影。
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,  # 在LoRA模块中使用的dropout率
    bias="none",  # 设置bias的使用方式，这里没有使用bias
)

### 使用get_peft_model函数和给定的配置来获取一个PEFT模型

In [23]:
peft_model = get_peft_model(model, config)

# 【核心修复】解决 bitsandbytes 和 PyTorch autocast 结合时的 c10::Half != float 报错问题
# 原因是 LayerNorm 被转化为 float32 后，经过 autocast 传入 Linear8bitLt 时，
# bitsandbytes 会将 forward 记录为 float32，但在 backward 时 PyTorch 传回的梯度是 float16，导致类型不匹配崩溃。
# 解决方法：通过 forward_pre_hook 强制将传入 8-bit 层的隐藏状态转回 float16，即可彻底解决此冲突！
import torch
import bitsandbytes as bnb

for module in peft_model.modules():
    if isinstance(module, bnb.nn.Linear8bitLt):
        module.register_forward_pre_hook(
            lambda m, args: (args[0].to(torch.float16), *args[1:])
        )

In [24]:
# 打印 LoRA 微调训练的模型参数
peft_model.print_trainable_parameters()

trainable params: 1,966,080 || all params: 1,545,271,040 || trainable%: 0.12723204856023188


## 模型训练

#### Seq2SeqTrainingArguments 训练参数

**关于设置训练步数和评估步数**

基于 epochs 设置：

```python
    num_train_epochs=3,  # 训练的总轮数
    evaluation_strategy="epoch",  # 设置评估策略，这里是在每个epoch结束时进行评估
    warmup_steps=50,  # 在训练初期增加学习率的步数，有助于稳定训练
```

基于 steps 设置：

```python
    max_steps=100, # 训练总步数
    evaluation_strategy="steps", 
    eval_steps=25, # 评估步数
```

In [25]:
from transformers import Seq2SeqTrainingArguments

# 设置序列到序列模型训练的参数
training_args = Seq2SeqTrainingArguments(
    overwrite_output_dir=True,
    output_dir=model_dir,  # 指定模型输出和保存的目录
    per_device_train_batch_size=batch_size,  # 每个设备上的训练批量大小
    learning_rate=1e-3,  # 学习率
    num_train_epochs=1,  # 训练的总轮数
    evaluation_strategy="epoch",  # 设置评估策略为按步数评估
    # eval_steps=2,  # 每 2 步评估一次
    logging_strategy="steps",  # 日志记录策略为按步数
    logging_steps=1000,  # 每 1 步记录一次日志
    save_strategy="epoch",  # 模型保存策略改为按 epoch
    load_best_model_at_end=True,  # 训练结束后加载最佳模型
    metric_for_best_model="eval_loss",  # 使用验证损失来选择最佳模型
    greater_is_better=False,  # 损失越小越好
    warmup_steps=50,  # 在训练初期增加学习率的步数，有助于稳定训练
    fp16=True,  # 启用混合精度训练，可以提高训练速度，同时减少内存使用
    per_device_eval_batch_size=4,  # 每个设备上的评估批量大小 (降低评估时的显存消耗)
    generation_max_length=16,  # 生成任务的最大长度
    predict_with_generate=True,  # 评估时生成文本计算WER
    eval_accumulation_steps=1,  # 评估时每1步就将预测结果转移到CPU，极大节省GPU内存
    remove_unused_columns=False,  # 是否删除不使用的列，以减少数据处理开销
    label_names=["labels"],  # 指定标签列的名称，用于训练过程中
    report_to="tensorboard",  # 将训练日志报告到 Tensorboard
    logging_dir=f"{model_dir}/logs",  # Tensorboard 日志目录
)

### 添加准确率公式

In [26]:
import evaluate
import numpy as np
from transformers import Seq2SeqTrainer

# 加载 WER 和 CER 评估指标
wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    # 替换 -100 为 pad_token_id
    label_ids[label_ids == -100] = tokenizer.pad_token_id

    # 解码预测和标签
    pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    # 计算 WER 和 CER
    wer = 100 * wer_metric.compute(predictions=pred_str, references=label_str)
    cer = 100 * cer_metric.compute(predictions=pred_str, references=label_str)

    return {
        "wer": wer,
        "cer": cer,
    }

### 实例化 Seq2SeqTrainer 训练器

In [27]:
from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    args=training_args,
    model=peft_model,
    train_dataset=tokenized_common_voice["train"],
    eval_dataset=tokenized_common_voice["validation"],
    data_collator=data_collator,
    tokenizer=processor.feature_extractor,
     compute_metrics=compute_metrics,  # 添加这一行

)
peft_model.config.use_cache = False

In [28]:
trainer.train()

/data00/home/zhanguo/miniforge3/envs/py310/lib/python3.10/site-packages/torch/utils/checkpoint.py:464: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.4 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  warnings.warn(
/data00/home/zhanguo/miniforge3/envs/py310/lib/python3.10/site-packages/torch/utils/checkpoint.py:91: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(


Epoch,Training Loss,Validation Loss,Wer,Cer
1,No log,3.496957,105.312500,62.799043


TrainOutput(global_step=7, training_loss=4.179028919764927, metrics={'train_runtime': 562.1521, 'train_samples_per_second': 1.138, 'train_steps_per_second': 0.012, 'total_flos': 1.36064139264e+18, 'train_loss': 4.179028919764927, 'epoch': 1.0})

### 测试集评估

In [29]:
# 评测
tokenized_test=tokenized_common_voice["test"]
# 在测试集上评估
test_results = trainer.evaluate(tokenized_test)
print("Test Results:")
print(f"Test Loss: {test_results['eval_loss']:.4f}")
print(f"Test WER: {test_results['eval_wer']:.2f}%")
print(f"Test CER: {test_results['eval_cer']:.2f}%")


Test Results:
Test Loss: 3.0681
Test WER: 101.56%
Test CER: 62.80%


### 保存 LoRA 模型(Adapter)

In [30]:
trainer.save_model(model_dir)

In [31]:
peft_model.eval()

PeftModel(
  (base_model): LoraModel(
    (model): WhisperForConditionalGeneration(
      (model): WhisperModel(
        (encoder): WhisperEncoder(
          (conv1): Conv1d(80, 1280, kernel_size=(3,), stride=(1,), padding=(1,))
          (conv2): Conv1d(1280, 1280, kernel_size=(3,), stride=(2,), padding=(1,))
          (embed_positions): Embedding(1500, 1280)
          (layers): ModuleList(
            (0-31): 32 x WhisperEncoderLayer(
              (self_attn): WhisperSdpaAttention(
                (k_proj): Linear8bitLt(in_features=1280, out_features=1280, bias=False)
                (v_proj): lora.Linear8bitLt(
                  (base_layer): Linear8bitLt(in_features=1280, out_features=1280, bias=True)
                  (lora_dropout): ModuleDict(
                    (default): Dropout(p=0.05, inplace=False)
                  )
                  (lora_A): ModuleDict(
                    (default): Linear(in_features=1280, out_features=4, bias=False)
                  )
            

## 模型推理（可能需要重启 Notebook）

**再次加载模型会额外占用显存，如果显存已经达到上限，建议重启 Notebook 后再进行以下操作**


In [32]:
model_dir = "../models/whisper-large-v2-asr-int8"

language = "Chinese (China)"
language_abbr = "zh-CN"
language_decode = "chinese"
task = "transcribe"


### 使用 `PeftModel` 加载 LoRA 微调后 Whisper 模型

使用 `PeftConfig` 加载 LoRA Adapter 配置参数，使用 `PeftModel` 加载微调后 Whisper 模型

In [33]:
from transformers import AutoModelForSpeechSeq2Seq, AutoTokenizer, AutoProcessor
from peft import PeftConfig, PeftModel

peft_config = PeftConfig.from_pretrained(model_dir)

base_model = AutoModelForSpeechSeq2Seq.from_pretrained(
    peft_config.base_model_name_or_path, load_in_8bit=True, device_map="auto"
)

peft_model = PeftModel.from_pretrained(base_model, model_dir)

/data00/home/zhanguo/miniforge3/envs/py310/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [34]:
tokenizer = AutoTokenizer.from_pretrained(peft_config.base_model_name_or_path, language=language, task=task)
processor = AutoProcessor.from_pretrained(peft_config.base_model_name_or_path, language=language, task=task)
feature_extractor = processor.feature_extractor

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


### 使用 Pipeline API 部署微调后 Whisper 实现中文语音识别任务

In [35]:
test_audio = "data/audio/test_zh.flac"

In [36]:
from transformers import AutomaticSpeechRecognitionPipeline

pipeline = AutomaticSpeechRecognitionPipeline(model=peft_model, tokenizer=tokenizer, feature_extractor=feature_extractor)

forced_decoder_ids = processor.get_decoder_prompt_ids(language=language_decode, task=task)

In [37]:
import torch

with torch.cuda.amp.autocast():
    text = pipeline(test_audio, max_new_tokens=255)["text"]

/data00/home/zhanguo/miniforge3/envs/py310/lib/python3.10/site-packages/bitsandbytes/autograd/_functions.py:322: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


In [38]:
text

' This is a test for the V2 model of WhisperLine.'

## Homework

1. 使用完整的数据集训练，对比 Train Loss 和 Validation Loss 变化。训练完成后，使用测试集进行模型评估.
2. [Optional]使用其他语种（如：德语、法语等）的数据集进行微调训练，并进行模型评估模型评估。